# Analytical null integration — summary

**Task** `analytical-null-md`. Commands, full output, and caveats:
`verification.md`.

## What this PR changes

The web query's null (`query_metapath_z`, `src/multi_dwpc_query.py`) is
replaced: instead of scoring a gene set against `b` random gene subsets
drawn from the whole gene universe (Monte Carlo, blind to gene degree), it
now uses the exact moments of the same resampling scheme over a
capacity-stratified partition (`analytical_gene_set_z`,
`src/analytical_null.py`, backed by `hetnetex_md.exact_resampling_moments`)
— deterministic, no `b` to tune, degree-honest.

## Hypotheses (as approved, `design.md`)

1. "The example query's analytical z correlates with the Monte-Carlo z."
2. "Seed-to-seed MC spread is visible at default `b`; the analytical value
   has none."
3. "Query latency drops by an order of magnitude or more."


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

BASE = Path.cwd()
if not (BASE / "tables").exists():
    BASE = Path("docs/tasks/analytical-null-md")
TABLES = BASE / "tables"
FIGURES = BASE / "figures"


In [ ]:
b_sweep_summary = pd.read_csv(TABLES / "b_sweep_summary.csv")
display(b_sweep_summary)


In [ ]:
Image(filename=str(FIGURES / "b_sweep_agreement.png"))


**The analytical value is the exact limit of the Monte-Carlo null.** The
Monte-Carlo estimate of the *identical* stratified null converges
monotonically to the analytical value as `B` grows (rho 0.9752 at `B=20` ->
0.9922 -> 0.9979 -> 0.9995 at `B=10,000`, mean of 3 seeds; table above) —
the analytical result is this null's exact `B -> infinity` limit. The
*old*, unstratified null does not converge even at `b=10,000` (rho 0.283,
table above): this PR changes both the null's definition (stratified vs.
not) and its computation (closed form vs. sampled), and this figure is what
separates the two.


In [ ]:
comparison = pd.read_csv(TABLES / "per_metapath_comparison.csv")
display(comparison.head())


In [ ]:
Image(filename=str(FIGURES / "old_vs_new_z_scatter.png"))


In [ ]:
Image(filename=str(FIGURES / "mc_seed_spread.png"))


**What it does to query results.** Same query, both nulls, 52 metapaths
(table above; full table `tables/per_metapath_comparison.csv`). The
seed-spread figure makes determinism visible: the analytical z is a single
point where the old null scatters across seeds (up to roughly 19 z units at
`b=20`). The scatter figure shows the shift itself: the analytical z sits
generally at or below the Monte-Carlo z, most visibly at the high end of
the ranking.


In [ ]:
Image(filename=str(FIGURES / "rank_agreement.png"))


**Rank agreement.** Hypothesis: the analytical z correlates with the
Monte-Carlo z. Measured: Spearman rho = 0.2605 (p = 0.0877, n = 44) —
weakly supported, not significant at alpha = 0.05. The mid-ranking
reordering visible in the figure is consistent with a degree-confound
explanation (interpretation, not tested directly by this run).


In [ ]:
timing = pd.read_csv(TABLES / "timing.csv")
display(timing)


In [ ]:
Image(filename=str(FIGURES / "timing_comparison.png"))


In [ ]:
Image(filename=str(FIGURES / "b_sweep_tradeoff.png"))


**Cost.** Hypothesis: latency drops by an order of magnitude or more.
Measured: not met at the app's `b=20` default — end-to-end latency is
disk-dominated and roughly unchanged (103 s vs 85 s, median of 3 runs;
table above). Matching the analytical precision by raising `B` instead is
expensive: the same-null MC sweep at `B=10,000` costs roughly 173 s
(3-seed total) against the analytical closed form's 6.4 ms — roughly
27,000x on that 3-seed total, roughly 9,000x per individual seed
(`verification.md`, "Agreement and cost versus B").


## Scorecard and onward path

- **Deterministic** — met (`verification.md`, "Positive control").
- **Seed-to-seed MC spread visible; analytical has none** — met (figure
  above).
- **Analytical z correlates with the Monte-Carlo z** — weakly supported
  (rho 0.26, p = 0.088).
- **Query latency drops by an order of magnitude or more** — not met
  (103 s vs 85 s; roughly 27,000x MC cost to match analytical precision).

Commands, full output, and caveats: [`verification.md`](verification.md).
Claim-by-claim audit: [`audit.md`](audit.md). Decision lineage, including
the 2026-09-02 reversal that keeps hypotheses stated as approved:
[`decisions.md`](decisions.md).
